## 1. Data Loading and Exploration

Loading the raw dataset and performing an initial inspection to 
understand the structure, data types, and any data quality issues 
before cleaning.

In [16]:
import pandas as pd

df = pd.read_excel('Effort .xlsx')
df.shape
df.info()
df.describe() 
df.isnull().sum()



<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Workload    5 non-null      str    
 1   Unnamed: 1  14 non-null     str    
 2   Jan         14 non-null     float64
 3   Feb         14 non-null     float64
 4   Mar         14 non-null     float64
 5   Apr         14 non-null     float64
 6   May         14 non-null     float64
 7   June        14 non-null     float64
 8   July        14 non-null     float64
 9   August      14 non-null     float64
 10  Sept        14 non-null     float64
 11  Oct         14 non-null     float64
 12  Nov         14 non-null     float64
 13  Dec         14 non-null     float64
dtypes: float64(12), str(2)
memory usage: 2.2 KB


Workload      14
Unnamed: 1     5
Jan            5
Feb            5
Mar            5
Apr            5
May            5
June           5
July           5
August         5
Sept           5
Oct            5
Nov            5
Dec            5
dtype: int64

### Initial Observations
- Dataset includes two distinct tables which need to be seperated
- There are blank rows used to segregate data in excel format
- Team names are only populated on the first row 
  of each team section — subsequent rows show NaN and require forward filling
- Team naming is inconsistent between the two tables, requires standardisation 
  before the tables can be joined
- All numeric values are stored as floats and should be 
  converted to integers
- Month values are stored as abbreviated text strings with inconsistent formatting and no year reference — these will 
  be converted to datetime format to enable calculations


## 2. Data Cleaning & Transformation

### Overview
The raw data required several cleaning and transformation steps before it 
could be loaded into a SQL database for analysis. The following section 
documents each step taken and the rationale behind it.

### Steps Performed

**Step 1: Split into Two Tables**
The raw Excel sheet contained two embedded tables — Workload and Manpower. 
These were separated into two distinct DataFrames for independent cleaning 
before being joined later in SQL.

**Step 2: Remove Empty Rows**
Blank rows used as visual separators in Excel were dropped as they contained 
no meaningful data.

**Step 3: Forward Fill Team Names**
Team names were only populated on the first row of each section. The `ffill()` 
method was used to propagate team names downward to all associated rows.

**Step 4: Drop Redundant Total Rows**
Rows containing aggregated totals (Total) were removed as these are 
derivable from New and Repeated values and would skew analysis if included.

**Step 5: Add Clinic Column**
A new `clinic` column was created to distinguish which clinic each team 
belongs to — the suffix A and B in the team name denotes the clinic 
(e.g. Team 1A and Team 1B both belong to Team 1 but operate in 
different clinics). This was extracted from the team name for use 
as a separate dimension in analysis.

**Step 6: Reshape from Wide to Long Format**
Both tables were reshaped from wide format (months as columns) to long 
format (months as rows) using `pd.melt()`. This is required for proper 
database storage and analysis.

**Step 7: Standardise Month Values to Datetime**
Month values were stored as inconsistent abbreviated text strings 
(e.g. "Jan", "June", "August"). These were mapped to a standardised 
datetime format (YYYY-MM-01) to enable time-based calculations in SQL 
and correct time axis rendering in Power BI.

**Step 8: Convert Numeric Columns to Integer**
All patient count and headcount values were stored as floats (e.g. 2524.0) 
due to the presence of NaN values in the raw data. These were converted 
to integers after cleaning.

In [113]:
# Manually split into two separate DataFrames based on row index
workload_raw = df.iloc[:17].copy()
manpower_raw = df.iloc[17:].copy()

# Drop blank separator rows — identified by null values in the patient type column
workload_raw = workload_raw.dropna(subset=['Unnamed: 1'])

# Forward fill team names as they are only populated on the first row of each section
workload_raw['Workload'] = workload_raw['Workload'].ffill()

#renaming columns for clarity
workload_raw = workload_raw.rename(columns={
    'Workload': 'team',
    'Unnamed: 1': 'patient_type'
})

# Drop redundant total rows as they are derivable from New + Repeated and would skew analysis
workload_raw = workload_raw[workload_raw['patient_type'] != 'Total']

# Extract clinic letter (A/B) and standardise team name (Team 1A → Team 1)
workload_raw['clinic'] = workload_raw['team'].str.extract(r'(\w)$')
workload_raw['team'] = workload_raw['team'].str.extract(r'(Team \d)')

# Melt from wide to long format — converts month columns into rows
# Manually have to check the col names due to inconsistent naming 
workload_raw = workload_raw.rename(columns={
    'June': '2025-06-01',
    'July': '2025-07-01',
    'August': '2025-08-01',
    'Sept': '2025-09-01',
    'Jan': '2025-01-01',
    'Feb': '2025-02-01',
    'Mar': '2025-03-01',
    'Apr': '2025-04-01',
    'May': '2025-05-01',
    'Oct': '2025-10-01',
    'Nov': '2025-11-01',
    'Dec': '2025-12-01'
})

months = [col for col in workload_raw.columns if col not in ['team', 'clinic', 'patient_type']]

workload_clean = workload_raw.melt(
    id_vars=['team', 'clinic', 'patient_type'],
    value_vars=months,
    var_name='month',
    value_name='patient_count'
)

#Standardizing data types
workload_clean['month'] = pd.to_datetime(workload_clean['month'])
workload_clean['patient_count'] = workload_clean['patient_count'].astype(int)


# Repeat process for Manpower Data
manpower_raw = manpower_raw.drop(columns=['Workload'])
manpower_raw = manpower_raw.rename(columns={'Unnamed: 1': 'team'})
manpower_raw = manpower_raw.rename(columns={
    'June': '2025-06-01',
    'July': '2025-07-01',
    'August': '2025-08-01',
    'Sept': '2025-09-01',
    'Jan': '2025-01-01',
    'Feb': '2025-02-01',
    'Mar': '2025-03-01',
    'Apr': '2025-04-01',
    'May': '2025-05-01',
    'Oct': '2025-10-01',
    'Nov': '2025-11-01',
    'Dec': '2025-12-01'
})

months = [col for col in manpower_raw.columns if col not in ['team']]

manpower_clean = manpower_raw.melt(
    id_vars=['team'],
    value_vars=months,
    var_name='month',
    value_name='headcount'
)

manpower_clean['month'] = pd.to_datetime(manpower_clean['month'])
manpower_clean['headcount'] = manpower_clean['headcount'].astype(int)



## 3. SQL Analysis

### Overview
The cleaned data is loaded into a local SQLite database to perform structured 
analysis using SQL. This reflects a real-world workflow where an analyst 
queries an existing database to extract and analyse data before presenting 
findings to stakeholders.

SQLite was chosen for its simplicity and portability — no server setup is 
required and the database is stored as a local file, making the project 
fully reproducible.

### Approach
Rather than performing all calculations in Power BI, SQL is used here to 
demonstrate how an analyst would query a database to draw conclusions from 
data. Each query is accompanied by a written interpretation of the results.

### Database Schema
Two tables are loaded into the database reflecting a simple relational model:

**workload** — patient visit counts by team, clinic, patient type and month
| Column | Type | Description |
|--------|------|-------------|
| team | TEXT | Team identifier (Team 1, Team 2) |
| clinic | TEXT | Clinic identifier (A or B) |
| patient_type | TEXT | New or Repeated |
| month | DATE | First day of each month (YYYY-MM-01) |
| patient_count | INTEGER | Number of patient visits |

**manpower** — monthly headcount by team
| Column | Type | Description |
|--------|------|-------------|
| team | TEXT | Team identifier (Team 1, Team 2) |
| month | DATE | First day of each month (YYYY-MM-01) |
| headcount | INTEGER | Number of staff |

### Queries
1. **Total patient volume by team and clinic** — overall workload distribution 
   across teams and clinics
2. **New vs Repeated patient breakdown** — patient type ratio by team 
   to understand returning patient behaviour
3. **Monthly trends** — total patient volume by month to identify 
   seasonal patterns
4. **Patients per staff member** — workload and manpower tables joined 
   to calculate staff workload efficiency by team and month
5. **Month-over-month change** — window functions used to calculate 
   volume change between consecutive months to identify growth or decline

### 3.1 Load Data into SQLite Database
The cleaned DataFrames are loaded into a local SQLite database. This creates 
a persistent database file which Power BI will later connect to.

In [52]:
import sqlite3

# Connect to SQLite database — creates workload.db file if it doesn't exist
conn = sqlite3.connect('workload.db')

# Load cleaned DataFrames into database tables
workload_clean.to_sql('workload', conn, if_exists='replace', index=False)
manpower_clean.to_sql('manpower', conn, if_exists='replace', index=False)

24

### 3.2 Total Patient Volume by Team and Clinic
The first query examines the overall workload distribution across teams and 
clinics to identify which teams carry the highest patient volume over the 
full year.

In [55]:
query1 = """
SELECT 
    team,
    SUM(patient_count) AS total_patients
FROM workload
GROUP BY team
ORDER BY total_patients DESC
"""

result1 = pd.read_sql(query1, conn)
print(result1)

query2 = """
SELECT 
    clinic,
    SUM(patient_count) AS total_patients
FROM workload
GROUP BY clinic
ORDER BY total_patients DESC
"""

result2 = pd.read_sql(query2, conn)
print(result2)

     team  total_patients
0  Team 1          169424
1  Team 2           56924
  clinic  total_patients
0      A          179533
1      B           46815


### Conclusion

**By Team:**
Team 1 carries the majority of the annual patient volume at 169,424 visits 
(75%) compared to Team 2 at 56,924 visits (25%), suggesting Team 1 operates 
at approximately three times the workload of Team 2.

**By Clinic:**
Clinic A accounts for the majority of patient volume at 179,533 visits (79%) 
compared to Clinic B at 46,815 visits (21%). This indicates that Clinic A 
is significantly busier regardless of team, and may warrant closer attention 
when analysing staff workload efficiency.

### 3.3 New vs Repeated Patient Breakdown
This query examines the ratio of new to repeated patients by team to 
understand returning patient behaviour and demand patterns across teams.

In [57]:
query3 = """
SELECT 
    team,
    patient_type,
    SUM(patient_count) AS total_patients,
    ROUND(100.0 * SUM(patient_count) / SUM(SUM(patient_count)) OVER (PARTITION BY team), 1) AS percentage
FROM workload
GROUP BY team, patient_type
"""

result3 = pd.read_sql(query3, conn)
print(result3)

     team patient_type  total_patients  percentage
0  Team 1          New           42670        25.2
1  Team 1          Rep          126754        74.8
2  Team 2          New           23547        41.4
3  Team 2          Rep           33377        58.6


### Conclusion

Both teams show a higher proportion of repeated patients than new patients, 
indicating a stable returning patient base across the organisation.

Team 1 has a significantly higher repeated patient ratio at 74.8% compared 
to 25.2% new patients, suggesting a well-established patient base with strong 
retention. Team 2 shows a more balanced split with 58.6% repeated and 41.4% 
new patients, indicating a relatively higher intake of new patients compared 
to Team 1.

The higher new patient ratio in Team 2 may reflect a newer or growing service, 
or differences in the nature of care provided between the two teams.

### 3.4 Monthly Trends
This query examines total patient volume by month across all teams and clinics 
to identify seasonal patterns and peak periods throughout the year.

In [65]:
query4 = """
SELECT 
    strftime('%Y-%m', month) AS month,
    SUM(patient_count) AS total_patients
FROM workload
GROUP BY month
ORDER BY month
"""

result4 = pd.read_sql(query4, conn)
print(result4)

      month  total_patients
0   2025-01           20279
1   2025-02           17253
2   2025-03           18174
3   2025-04           18504
4   2025-05           18385
5   2025-06           16696
6   2025-07           20264
7   2025-08           19398
8   2025-09           18796
9   2025-10           20003
10  2025-11           19708
11  2025-12           18888


### Conclusion
January records the highest patient volume at 20,279 visits while June 
records the lowest at 16,696 visits. While SQL is useful for extracting 
and aggregating this data, identifying trends and seasonal patterns is 
better explored visually — monthly trends will be examined in greater 
detail in the Power BI dashboard.

### 3.5 Patients per Staff Member
This query joins the workload and manpower tables to calculate the daily average 
number of patients handled per staff member by team across the year, providing 
a measure of staff workload efficiency. It is taken that there are 22 working days in a month.

In [ ]:
query5 = '''
SELECT 
    w.team,
    AVG(m.headcount) AS Avg_headcount,
    ROUND(SUM(patient_count)/(AVG(m.headcount)*COUNT(DISTINCT w.month)*22),2) AS Patients_Per_Staff_Per_WorkingDay
FROM workload w
JOIN manpower m 
    ON w.team = m.team
    AND w.month = m.month
GROUP BY w.team
'''

result5 = pd.read_sql(query5, conn)
print(result5)

     team  Avg_headcount  Patients_Per_Staff_Per_WorkingDay
0  Team 1     104.083333                               6.17
1  Team 2      35.916667                               6.00


### Conclusion
Both teams show a remarkably similar daily workload per staff member — 
Team 1 at 6.17 patients per staff per working day and Team 2 at 6.00, 
suggesting that despite the significant difference in total patient volume, 
staffing is proportionally allocated across both teams.

This consistency indicates effective workforce planning, where headcount 
scales appropriately with patient demand. Monthly variation in daily 
workload will be explored further in the Power BI dashboard.

### 3.6 Month-over-Month Change
This query uses a window function to calculate the change in total patient 
volume between consecutive months, identifying periods of growth or decline 
in patient demand throughout the year.

In [ ]:
query6 = '''
WITH monthly_visits AS (
    SELECT
        month,
        SUM(patient_count) AS total
    FROM workload
    GROUP BY month)
SELECT
    month,
    ROUND(100.0*(total-(LAG(total) OVER(ORDER BY month)))/(LAG(total) OVER(ORDER BY month)),2) AS mom_change
FROM monthly_visits
'''

result6 = pd.read_sql(query6, conn)
print(result6)

                  month  mom_change
0   2025-01-01 00:00:00         NaN
1   2025-02-01 00:00:00      -14.92
2   2025-03-01 00:00:00        5.34
3   2025-04-01 00:00:00        1.82
4   2025-05-01 00:00:00       -0.64
5   2025-06-01 00:00:00       -9.19
6   2025-07-01 00:00:00       21.37
7   2025-08-01 00:00:00       -4.27
8   2025-09-01 00:00:00       -3.10
9   2025-10-01 00:00:00        6.42
10  2025-11-01 00:00:00       -1.47
11  2025-12-01 00:00:00       -4.16


### Conclusion
January has no previous month to compare against so MoM change is null 
for that month. The most notable changes are July which saw a sharp 
increase of 21.37% from June, and February which saw the steepest decline 
at -14.92% from January. June also recorded a significant drop of -9.19% 
from May, consistent with the earlier finding that June is the lowest 
volume month of the year.

Overall volume fluctuates moderately month to month but without a clear 
directional trend, suggesting demand is cyclical rather than growing or 
declining.

### 3.7 Export Tables for Power BI

The cleaned workload and manpower tables are exported as separate CSV files 
for Power BI. 

In [112]:
# Export two separate clean tables for Power BI
workload_clean.to_csv('workload.csv', index=False)
manpower_clean.to_csv('manpower.csv', index=False)

conn.close()
print("Exported workload.csv and manpower.csv")

Exported workload.csv and manpower.csv
